# LLM Systems Engineer Assignment

**Name:** Khushi Mundhada  

## Overview
This assignment explores practical system-level interaction with Large Language Models (LLMs).  
The focus is on understanding model architecture, performing efficient fine-tuning using LoRA,  and experimenting with model composition techniques.

The implementation includes:
- Model architecture parsing
- LoRA-based fine-tuning on a real dataset
- Model merging and output comparison

## Setup and Installation
Installing required libraries and dependencies.

In [ ]:
!pip install transformers datasets peft accelerate bitsandbytes

## Model Loading

I used GPT-2 as a lightweight and interpretable baseline model for experimentation.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


## Task 1: Model Architecture Parsing

The goal is to extract and represent the internal structure of the model in a clean and hierarchical format.

### Parser Implementation

In [ ]:
def parse_model(model):
    structure = {}
    for name, module in model.named_children():
        structure[name] = str(type(module))
        sub_layers = {}
        for sub_name, sub_module in module.named_children():
            sub_layers[sub_name] = str(type(sub_module))
        if sub_layers:
            structure[name] = sub_layers
    return structure

### Parsed Output

In [ ]:
import json
parsed = parse_model(model)
print(json.dumps(parsed, indent=2))

{
  "transformer": {
    "wte": "<class 'torch.nn.modules.sparse.Embedding'>",
    "wpe": "<class 'torch.nn.modules.sparse.Embedding'>",
    "drop": "<class 'torch.nn.modules.dropout.Dropout'>",
    "h": "<class 'torch.nn.modules.container.ModuleList'>",
    "ln_f": "<class 'torch.nn.modules.normalization.LayerNorm'>"
  },
  "lm_head": "<class 'torch.nn.modules.linear.Linear'>"
}


### Deep Parsing (Recursive)

In [ ]:
def deep_parse(module, depth=0, max_depth=3):
    if depth > max_depth:
        return str(type(module))

    structure = {}
    for name, sub_module in module.named_children():
        structure[name] = deep_parse(sub_module, depth+1, max_depth)

    return structure

### Observations

- GPT-2 consists of embedding layers, transformer blocks, and a final linear head.
- Each transformer block includes attention and MLP sublayers.
- The structure is repetitive, making it suitable for hierarchical parsing.

## Task 2: Fine-Tuning with LoRA

Ifine-tuneD GPT-2 using Parameter Efficient Fine-Tuning (LoRA)  
on a subset of the IMDB dataset.

### Dataset Preparation

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset("imdb", split="train[:5%]")

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

### LoRA Configuration

Note: GPT-2 uses Conv1D layers instead of standard Linear layers, requiring `fan_in_fan_out=True` for correct LoRA application.


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    fan_in_fan_out=True
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.to("cuda")

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


PeftModel(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_

### Baseline Output (Before Fine-Tuning)

We first evaluate the base model's output before applying any fine-tuning.
This serves as a baseline for comparison.

In [ ]:
prompt = "This movie was"

inputs = tokenizer(prompt, return_tensors="pt", padding=True).to("cuda")

output = model.generate(
    **inputs,
    max_length=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
)

print("Before Fine-tuning:\n", tokenizer.decode(output[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Before Fine-tuning:
 This movie was filmed in the summer of 2011. It is believed to have been shot around May 2015, before it officially became available for theatrical release on DVD and Blu-ray Disc.[9]





### Training

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="no",
    fp16=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Step,Training Loss
10,4.285596
20,4.475076
30,4.145488
40,4.473566
50,4.394091
60,4.279626
70,4.098629
80,4.047780
90,4.191645
100,4.099085


TrainOutput(global_step=626, training_loss=3.9282244630515004, metrics={'train_runtime': 59.5031, 'train_samples_per_second': 42.015, 'train_steps_per_second': 10.52, 'total_flos': 163873751040000.0, 'train_loss': 3.9282244630515004, 'epoch': 2.0})

### Training Observations

- Training loss decreases over iterations, indicating learning.
- LoRA enables efficient fine-tuning without updating all parameters.
- The model adapts slightly to sentiment patterns in the dataset.

## Task 3: Model Composition

We merge the LoRA adapter back into the base model  
to create a standalone fine-tuned model.

In [ ]:
model = model.merge_and_unload()

### Output After Fine-Tuning

In [ ]:
prompt = "This movie was"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    early_stopping=True
)

print("After Fine-tuning:\n", tokenizer.decode(output[0], skip_special_tokens=True))

After Fine-tuning:
 This movie was a hit. Theatrical release for the film with an overall budget of $11 million (with over 4,000 screens). It is one in two films to be released this year and it has already been nominated by numerous major studios including New Line Cinema , Universal Pictures Television / Fox Film 2 , Warner Bros., Lionsgate Productions .


### Comparison, Observations and Conclusion

- Before fine-tuning: The model produced generic, repetitive text that was not aligned with movie reviews or sentiment.

- After fine-tuning: The output shows a clear shift toward movie-related and review-style content (e.g., references to theatrical release, budget, studios), indicating that the model has partially adapted to the IMDB dataset.

- This suggests that fine-tuning introduced a domain-specific bias, improving fluency and making the text more contextually relevant.

However:
- The output still lacks strong sentiment expression (clear positive/negative stance).
- Some details are hallucinated or loosely connected.
- Coherence is improved but not consistent throughout.

This demonstrates that:
- LoRA fine-tuning can effectively steer generation style with very few trainable parameters.
- However, meaningful control over model behavior requires larger datasets, longer training, and more capable base models.

Overall, the experiment highlights both the strengths and limitations of lightweight fine-tuning in shaping LLM outputs.

This highlights how even lightweight fine-tuning can steer generation style, though not necessarily guarantee semantic correctness.